# 🚀 Real-Time Yahoo Finance Multi-Company LSTM Predictor
### Train ONE Universal Model on Live Stock Data (NIFTY 50, US Tech, Crypto, or Custom Tickers)

This notebook enables you to:
1. **Stream fresh, split-adjusted market data directly from Yahoo Finance API** (no old/corrupted datasets, no manual CSV uploading).
2. **Train ONE unified PyTorch LSTM** across multiple companies simultaneously (e.g. RELIANCE, TCS, INFY, HDFC, or US stocks like NVDA, AAPL, TSLA).
3. **Use Window-Relative Scaling** to eliminate multi-decade distortion and achieve ultra-low test MAPE (~1-2%).
4. **Click or select any company from an interactive dropdown** to immediately see its live price, test prediction curve, and 5-candle forecast trajectory ahead.

---
## 1. Hardware Check & Environment Setup
Ensure GPU acceleration is active: **Runtime** → **Change runtime type** → select **T4 GPU**.

In [ ]:
# Verify GPU hardware
!nvidia-smi

# Install / verify required packages
!pip install -q yfinance torch pandas numpy matplotlib scikit-learn ipywidgets

---
## 2. Clone GitHub Repository
Clones the codebase containing the LSTM model and feature engineering pipeline.

In [ ]:
import os

REPO_URL = "https://github.com/Raghava-sensai/stock-lstm-prediction.git"
REPO_DIR = "stock-lstm-prediction"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

print(f"\n[OK] Active Working Directory: {os.getcwd()}")

---
## 3. Choose Tickers & Download Fresh Market Data via Yahoo Finance
Select a preset list or type your own custom tickers (comma-separated).

In [ ]:
#@title 🌐 Select Tickers & Time Horizon { run: "auto" }
TICKER_SELECTION = "Top Indian NIFTY 10" #@param ["Top Indian NIFTY 10", "US Tech Giants", "Top Crypto Assets", "Custom Tickers"]
custom_symbols = "RELIANCE.NS, TCS.NS, INFY.NS, HDFCBANK.NS, TATAMOTORS.NS" #@param {type:"string"}
period = "2y" #@param ["6mo", "1y", "2y", "5y"]
interval = "1d" #@param ["1d", "1h"]

if TICKER_SELECTION == "Top Indian NIFTY 10":
    selected_symbols = [
        "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS",
        "SBIN.NS", "BHARTIARTL.NS", "ITC.NS", "LT.NS", "KOTAKBANK.NS"
    ]
elif TICKER_SELECTION == "US Tech Giants":
    selected_symbols = ["NVDA", "AAPL", "MSFT", "TSLA", "GOOGL", "AMZN", "META"]
elif TICKER_SELECTION == "Top Crypto Assets":
    selected_symbols = ["BTC-USD", "ETH-USD", "SOL-USD", "BNB-USD"]
else:
    selected_symbols = [s.strip().upper() for s in custom_symbols.split(",") if s.strip()]

print(f"[*] Selected {len(selected_symbols)} tickers:")
print("   ", ", ".join(selected_symbols))

---
## 4. Train ONE Universal Model on Live Yahoo Finance Data (GPU)
This trains a single PyTorch Stacked LSTM with **AdamW** across all selected tickers.
Uses **Window-Relative Scaling (Reversible Instance Normalization)** to achieve ultra-low test error (~1-2% MAPE).

In [ ]:
#@title ⚙️ Training Hyperparameters { run: "auto" }
epochs = 20 #@param {type:"integer"}
batch_size = 128 #@param [64, 128, 256, 512] {type:"raw"}
forecast_horizon = 5 #@param {type:"integer"}
seq_length = 24 #@param {type:"integer"}

symbols_arg = ",".join(selected_symbols)
output_dir = "./outputs/yahoo_multi_company"

!python yahoo_multi_company_lstm.py \
    --symbols "{symbols_arg}" \
    --period "{period}" \
    --interval "{interval}" \
    --epochs {epochs} \
    --batch_size {batch_size} \
    --seq_length {seq_length} \
    --forecast_horizon {forecast_horizon} \
    --output_dir "{output_dir}"

---
## 5. Multi-Company Performance Leaderboard
Inspect model performance across all tickers sorted by lowest test MAPE.

In [ ]:
import os
import pandas as pd
from IPython.display import display

leaderboard_path = "outputs/yahoo_multi_company/yahoo_multi_company_leaderboard.csv"
if os.path.exists(leaderboard_path):
    df_board = pd.read_csv(leaderboard_path)
    print(f"\n[+] Master Yahoo Finance Leaderboard ({len(df_board)} Tickers):")
    display(df_board)
else:
    print("[!] Run training in Section 4 first to generate leaderboard.")

---
## 6. 🎯 Interactive Stock Selector & Forecast Viewer
Pick **any ticker** from the interactive dropdown below to view:
1. **Actual vs Predicted Overlay**: Historical test set performance.
2. **Projected 5-Candle Trajectory Cone Ahead**: Live price path with Bullish/Bearish signal.

In [ ]:
import pickle
from yahoo_multi_company_lstm import view_yahoo_forecast, launch_interactive_widget

cache_file = "outputs/yahoo_multi_company/eval_results.pkl"
with open(cache_file, "rb") as f:
    eval_results = pickle.load(f)

# Launch interactive dropdown widget
launch_interactive_widget(eval_results)

---
### Direct Single-Stock Lookup (Colab Form Field)
Alternatively, type any ticker name below and run to render the forecast chart:

In [ ]:
#@title 📈 Quick Ticker Lookup { run: "auto" }
target_ticker = "RELIANCE.NS" #@param {type:"string"}

clean_ticker = target_ticker.strip().upper()
if clean_ticker in eval_results:
    view_yahoo_forecast(clean_ticker, eval_results, show_plot=True)
else:
    available = sorted(list(eval_results.keys()))
    print(f"[!] Ticker '{clean_ticker}' not found in trained set. Available tickers:\n  {available}")